In [2]:
import pandas as pd
df = pd.read_csv("day.csv")
df.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [3]:
df[["dteday","workingday","casual","registered","cnt"]].head()

,dteday,workingday,casual,registered,cnt
0,2011-01-01,0,331,654,985
1,2011-01-02,0,131,670,801
2,2011-01-03,1,120,1229,1349
3,2011-01-04,1,108,1454,1562
4,2011-01-05,1,82,1518,1600


In [4]:
# Number of rows and columns
print("Dataset shape:", df.shape)

# Missing values in each column
print("\nMissing values")
print(df.isna().sum())

# Number of completely duplicated rows
print("\nDuplicate rows:", df.duplicated().sum())

Dataset shape: (731, 16)

Missing values
instant       0
dteday        0
season        0
yr            0
mnth          0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0
dtype: int64

Duplicate rows: 0


In [5]:
# Convert the date column from text into datetime values
df["dteday"] = pd.to_datetime(df["dteday"])

# Put observations in chronological order
df = df.sort_values("dteday").reset_index(drop=True)

# Check the date range and repeated dates
print("Earliest date:", df["dteday"].min())
print("Latest date:", df["dteday"].max())
print("Repeated dates", df["dteday"].duplicated().sum())

Earliest date: 2011-01-01 00:00:00
Latest date: 2012-12-31 00:00:00
Repeated dates 0


In [6]:
# Training: all of 2011
train = df[
    (df["dteday"] >= "2011-01-01") &
    (df["dteday"] < "2012-01-01")
].copy()

# Validation: January-June 2012
validation = df[
    (df["dteday"] >= "2012-01-01") &
    (df["dteday"] < "2012-07-01")
].copy()

# Test: July-December 2012
test = df[
    (df["dteday"] >= "2012-07-01") &
    (df["dteday"] < "2013-01-01")
].copy()

print("Training rows:", len(train))
print("Validation rows:", len(validation))
print("Test rows:", len(test))

# Check that every row belongs to one of these groups
assert len(train) + len(validation) + len(test) == len(df)

Training rows: 365
Validation rows: 182
Test rows: 184


In [7]:
# Column we will use to predict retal demand
features = [
    "season", "mnth", "weekday", "holiday", "workingday",
    "weathersit", "temp", "hum", "windspeed"
]

# Training inputs and actual rental totals
X_train = train[features].copy()
y_train = train["cnt"].copy()

# Validation inputs and actual rental totals
X_validation = validation[features].copy()
y_validation = validation["cnt"].copy()

# Check their sizes
print("Training inputs:", X_train.shape)
print("training target:", y_train.shape)
print("Validation inputs:", X_validation.shape)
print("Validation target:", y_validation.shape)

X_train.head()

Training inputs: (365, 9)
training target: (365,)
Validation inputs: (182, 9)
Validation target: (182,)


,season,mnth,weekday,holiday,workingday,weathersit,temp,hum,windspeed
0,1,1,6,0,0,2,0.344167,0.805833,0.160446
1,1,1,0,0,0,2,0.363478,0.696087,0.248539
2,1,1,1,0,1,1,0.196364,0.437273,0.248309
3,1,1,2,0,1,1,0.200000,0.590435,0.160296
4,1,1,3,0,1,1,0.226957,0.436957,0.186900


In [8]:
# Learning the average daily rental count from training data only
training_mean = y_train.mean()

print("Training mean:", training_mean)

# Predict that same average for every validation day
baseline_predictions = pd.Series(
    training_mean,
    index=y_validation.index
)

# Compare the first five predictions with the actual counts
comparison = pd.DataFrame({
    "date": validation["dteday"],
    "actual": y_validation,
    "base_line": baseline_predictions
})

comparison.head()

Training mean: 3405.7616438356163


,date,actual,base_line
365,2012-01-01,2294,3405.761644
366,2012-01-02,1951,3405.761644
367,2012-01-03,2236,3405.761644
368,2012-01-04,2368,3405.761644
369,2012-01-05,3272,3405.761644


In [9]:
# Difference between actual and predicted rentals
errors = y_validation - baseline_predictions

# Mean Absolute Erros
baseline_mae = errors.abs().mean()

# Root Mean Squared Error
baseline_rmse = (errors ** 2).mean() ** 0.5

print(f"Baseline MAE: {baseline_mae:.2f} rentals/day")
print(f"Baseline RMSE: {baseline_rmse:.2f} rentals/day")

Baseline MAE: 2052.44 rentals/day
Baseline RMSE: 2449.89 rentals/day


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Category codes, not continuous measurements
categorical_features = [
    "season", "mnth", "weekday", "weathersit"
]

# Measurements and binary indicators
numeric_features = [
    "temp", "hum", "windspeed", "holiday", "workingday"
]

# Encode categories and keep the other inputs unchanged
preprocessor = ColumnTransformer([
    ("categories",
     OneHotEncoder(drop="first", handle_unknown="ignore"),
     categorical_features),

    ("numbers", "passthrough", numeric_features)
])

# Connect preprocessing to linear regression
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])

# Learn from training data only
linear_model.fit(X_train, y_train)

print("Linear regression training complete!")

Linear regression training complete!


In [11]:
# Predict rental counts for validation days
linear_predictions = pd.Series(
    linear_model.predict(X_validation),
    index=y_validation.index
)

# Calculate validation errors
linear_errors = y_validation - linear_predictions

linear_mae = linear_errors.abs().mean()
linear_rmse = (linear_errors ** 2).mean() ** 0.5

# Compare with our baseline
results = pd.DataFrame({
    "Model": ["Training-mean baseline", "Linear regression"],
    "Validation MAE": [baseline_mae, linear_mae],
    "Validation RMSE": [baseline_rmse, linear_rmse]
})

results.round(2)

,Model,Validation MAE,Validation RMSE
0,Training-mean baseline,2052.44,2449.89
1,Linear regression,2064.21,2248.70


In [13]:
# Compare both models on the same validation days
error_comparison = pd.DataFrame({
    "date": validation["dteday"],
    "actual": y_validation,
    "baseline_prediction": baseline_predictions,
    "linear_prediction": linear_predictions
})

# Calculate each model's absolute error
error_comparison["baseline_abs_error"] = (
    error_comparison["actual"] - error_comparison["baseline_prediction"]
).abs()

error_comparison["linear_abs_error"] = (
    error_comparison["actual"] - error_comparison["linear_prediction"]
).abs()

# Positive means linear regression made a smaller error
error_comparison["linear_improvement"] = (
    error_comparison["baseline_abs_error"]
    - error_comparison["linear_abs_error"]
)

error_comparison.head(10).round(2)

,date,actual,baseline_prediction,linear_prediction,baseline_abs_error,linear_abs_error,linear_improvement
365,2012-01-01,2294,3405.76,1719.05,1111.76,574.95,536.81
366,2012-01-02,1951,3405.76,1257.02,1454.76,693.98,760.78
367,2012-01-03,2236,3405.76,1159.63,1169.76,1076.37,93.39
368,2012-01-04,2368,3405.76,1089.28,1037.76,1278.72,-240.96
369,2012-01-05,3272,3405.76,1815.00,133.76,1457.00,-1323.23
370,2012-01-06,4098,3405.76,1998.62,692.24,2099.38,-1407.14
371,2012-01-07,4521,3405.76,2176.02,1115.24,2344.98,-1229.74
372,2012-01-08,3425,3405.76,1914.89,19.24,1510.11,-1490.87
373,2012-01-09,2376,3405.76,1253.23,1029.76,1122.77,-93.01
374,2012-01-10,3598,3405.76,1740.38,192.24,1857.62,-1665.38


In [14]:
one_day = pd.DataFrame({
    "date": validation["dteday"],
    "actual": y_validation,
    "baseline_prediction": baseline_predictions,
    "linear_prediction": linear_predictions
})

one_day.head(1).round(2)

,date,actual,baseline_prediction,linear_prediction
365,2012-01-01,2294,3405.76,1719.05


## Initial validation results

Training data: January–December 2011.
Validation data: January–June 2012.
Test data: July–December 2012; not evaluated.

| Model | Validation MAE | Validation RMSE |
|---|---:|---:|
| Training-mean baseline | 2052.44 | 2449.89 |
| Linear regression | 2064.21 | 2248.70 |

Linear regression had slightly worse MAE but lower RMSE.
Since MAE is the primary metric, linear regression did not
outperform the baseline on the primary metric.

Further error analysis is needed before explaining this difference.